In [1]:
import tensorflow as tf

In [2]:
#load pretrained model

base_model = tf.keras.applications.MobileNetV2(
    input_shape = (224, 224, 3),
    include_top = False,
    weights = "imagenet"
)

In [3]:
#Freeze the pretrained layer
base_model.trainable = False

In [4]:
# define the model input
inputs = tf.keras.Input(shape=(224, 224, 3))

In [6]:
#Apply data augmentation
from tensorflow.keras.layers import(
    RandomFlip,
    RandomRotation,
    RandomContrast,
    RandomTranslation
)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])

x = data_augmentation(inputs)

In [7]:
#Rescaling dataset

x = tf.keras.layers.Rescaling(
    scale=1./127.5,
    offset = -1
)(x)

In [8]:
#Pass images through MobileNetV2
x = base_model(x, training=False)

In [9]:
#Apply Global Average Pooling

x = tf.keras.layers.GlobalAveragePooling2D()(x)

In [10]:
#Add classification layer
outputs = tf.keras.layers.Dense(
    1,
    activation="sigmoid"
)(x)

1. Input layer
2. data augmentation layer
3. image processing
4. pretrained feature extractor
5. global average pooling
6. dense sigmoid output layer

In [11]:
mobilenet_model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs
)

In [12]:
mobilenet_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [13]:
#Compile the model

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [14]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights = True
)

In [ ]:
mobilenet_history = mobilenet_model.fit(
    train_ds,
    validation_data = val_ds,
    epochs = 15,
    callbacks=[early_stopping]
)